In [1]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, ClassifierMixin

# Define the Zero-Rule Baseline Classifier directly
class ZeroRuleClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.classes_, counts = np.unique(y, return_counts=True)
        self.majority_class_ = self.classes_[np.argmax(counts)]
        return self
    def predict(self, X):
        return np.array([self.majority_class_] * len(X))
    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)

print("Libraries and ZeroRuleClassifier loaded successfully!")

Libraries and ZeroRuleClassifier loaded successfully!


In [2]:
# Load from the correct folder path 'data_preparation/processed/'
train_path = "data_preparation/processed/train_cleaned.csv"
val_path = "data_preparation/processed/validation_cleaned.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

print("Cleaned Training Dataset Shape:", train_df.shape)
print("Cleaned Validation Dataset Shape:", val_df.shape)

Cleaned Training Dataset Shape: (19300, 530)
Cleaned Validation Dataset Shape: (1111, 530)


In [3]:
wap_cols = [col for col in train_df.columns if col.startswith('WAP')]

X_train_raw = train_df[wap_cols]
y_train_raw = train_df["LOCATION_CLASS"]

X_holdout = val_df[wap_cols]
y_holdout = val_df["LOCATION_CLASS"]

# Model-specific scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_holdout_scaled = scaler.transform(X_holdout)

X_train_df = pd.DataFrame(X_train_scaled, columns=wap_cols)
X_holdout_df = pd.DataFrame(X_holdout_scaled, columns=wap_cols)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_train_df, y_train_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_train_raw
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

Training feature shape: (15440, 520)
Testing feature shape: (3860, 520)


In [4]:
# Evaluate Zero-Rule Baseline (pass both X_train and y_train)
baseline = ZeroRuleClassifier()
baseline.fit(X_train, y_train)

baseline_acc = baseline.score(X_test, y_test)
print(f"Zero-Rule Baseline Test Accuracy: {baseline_acc * 100:.2f}%")

Zero-Rule Baseline Test Accuracy: 14.04%
